Español :
En esta versión del modelo he aplicado una mejora sencilla pero muy efectiva: he activado el parámetro class_weight='balanced' en el clasificador Random Forest.

Este ajuste hace que el algoritmo dé más peso a las clases que tienen menos ejemplos en el dataset. De este modo, evitamos que el modelo solo aprenda las clases mayoritarias y mejore su capacidad para predecir también las clases minoritarias.

 English :
In this improved version of the model, I applied a simple but powerful enhancement: I activated the parameter class_weight='balanced' in the Random Forest classifier.

This setting instructs the algorithm to give more importance to underrepresented classes. It helps the model avoid learning only from the majority class and improves its ability to predict minority classes.

In [4]:
# ================================================
# STEP 1: IMPORT LIBRARIES / IMPORTAR LIBRERÍAS
# ================================================

import pandas as pd
import numpy as np
import os

from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# ================================================
# STEP 2: LOAD DATASET / CARGAR DATASET
# ================================================

# Load Excel dataset (change path if needed)
# Cargar el archivo Excel (ajusta la ruta si es necesario)
file_path = r"C:\Users\hardr\OneDrive\Pictures\Datos adjuntos\Documents\Desktop\Python_scripts\ML_RIESGO_IMPAGO\src\data_sample\CAR4CASH_DATA.xlsx"
df = pd.read_excel(file_path)

# Clean column names (remove spaces)
# Limpiar nombres de columnas (eliminar espacios)
df.columns = df.columns.str.strip()

# ================================================
# STEP 3: CREATE TARGET VARIABLE / CREAR VARIABLE OBJETIVO
# ================================================

# Function to classify delay days into buckets
# Función para clasificar días de retraso en grupos de riesgo
def asignar_bucket(v):
    if pd.isna(v):
        return 'desconocido'
    try:
        v = int(v)
        if v >= 0:
            return 'no delay'
        elif v >= -5:
            return 'early'
        elif v >= -15:
            return 'medium'
        elif v >= -30:
            return 'late'
        elif v >= -60:
            return 'external'
        else:
            return 'realdebt'
    except:
        return 'desconocido'

df['Bucket'] = df['Vencimiento'].apply(asignar_bucket)
df = df[df['Bucket'] != 'desconocido']

# ================================================
# STEP 4: DEFINE FEATURES AND TARGET / DEFINIR X E Y
# ================================================

X = df.drop(columns=['Vencimiento', 'Bucket'])
y = df['Bucket']

cat_features = ['Código postal', 'Ciudad', 'Marca', 'Modelo', 'Partner', 'Genero']
num_features = ['Precio de compra en €', 'Plazo mensual', 'edad']

# ================================================
# STEP 5: DATA CLEANING / LIMPIEZA DE DATOS
# ================================================

# Convert categorical to string and fill missing
# Convertir categóricas a texto y rellenar valores nulos
for col in cat_features:
    X[col] = X[col].fillna("missing").astype(str)

# Convert numeric to float and detect errors as NaN
# Convertir numéricas a float y detectar errores como NaN
for col in num_features:
    X[col] = pd.to_numeric(X[col], errors='coerce')

# Optional check for missing values
print("Missing values per column:")
print(X.isnull().sum())

# ================================================
# STEP 6: CREATE PREPROCESSING PIPELINE
# ================================================

# Numeric pipeline: imputation + scaling
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Categorical pipeline: imputation + encoding
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine pipelines in ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_features),
    ('cat', cat_pipeline, cat_features)
])

# ================================================
# STEP 7: TRAIN-TEST SPLIT / DIVISIÓN ENTRENAMIENTO-PRUEBA
# ================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ================================================
# STEP 8: IMPROVED MODEL WITH CLASS BALANCING
# ================================================

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'  # Mejora para clases desbalanceadas
    ))
])

# ================================================
# STEP 9: TRAIN AND EVALUATE / ENTRENAR Y EVALUAR
# ================================================

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ================================================
# STEP 10: SAVE MODEL / GUARDAR MODELO
# ================================================

# Create output folder if it doesn't exist
os.makedirs("src/models", exist_ok=True)

# Save model using joblib
joblib.dump(pipeline, "src/models/modelo_riesgo_impago.pkl")
print("Model saved at: src/models/modelo_riesgo_impago.pkl")


Missing values per column:
ID                                                   0
Tipo                                                 0
Representante comercial                              0
Precio de compra                                     0
Moneda                                               0
IVA                                                  0
Precio de compra en €                                0
Firma del contrato                                   0
Fecha del principio de contrato                      0
Plazo mensual                                        0
Plazo del IVA                                        0
Cantidad de tasa de reserva en € IVA incluido        0
Cantidad de tasa de reserva en € sin IVA             0
Fecha del plazo siguiente                           37
Precio de mercado                                    0
Cociente precio de mercado/precio de compra (%)      0
Teléfono                                             0
Correo electrónico                    

c:\Users\hardr\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hardr\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\hardr\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo